In [52]:
import os
from dotenv import load_dotenv
from langsmith import Client
from langchain_groq import ChatGroq
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

In [76]:
load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
WEATHER_STACK_API_KEY = os.getenv("WEATHER_STACK_API_KEY")


if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add it to your .env file.")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing. Add it to your .env file.")
if not WEATHER_STACK_API_KEY:
    raise ValueError("WEATHER_STACK_API_KEY is missing. Add it to your .env file.")

In [54]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0,
    timeout=30
)

In [55]:
search_tool = TavilySearchResults(
    max_results=2,
    api_wrapper=TavilySearchAPIWrapper(tavily_api_key=TAVILY_API_KEY)
)

In [56]:
result = search_tool.invoke("Who won the FIFA World Cup 2026?")
print(result)

[{'title': 'FIFA World Cup 2026™ Final', 'url': 'https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/final', 'content': "Image 85\n\nImage 86: wordmark 1 1\n\nSpain claimed ultimate glory in front of a crowd of over 80,000 at New York New Jersey Stadium, as they were crowned FIFA World Cup™ champions on Sunday 19 July 2026.\n\nWorld Cup winners in 2010, Spain lifted the Trophy for a second time after defeating three-time champions Argentina 1-0 after extra-time in the Final.\n\nThe sides had only met once previously at the World Cup - a 2-1 Argentina win in 1966 - but Spain came out on top when it mattered in the biggest game of all 60 years later thanks to a goal from Ferran Torres.\n\n  \n\nImage 87Image 88Image 89Image 90Image 91Image 92Image 93Image 94Image 95Image 96Image 97Image 98Image 99Image 100Image 101Image 102Image 103Image 104Image 105Image 106Image 107 [...] Image 35: Spain's winner 🇪🇸new Spain's winner 🇪🇸Image 36: Ferran Torres 🏆 🇪🇸new Ferran Torres 🏆 🇪

In [57]:
result = llm.invoke("What year is it?")
print(result)

content='It’s the year\u202f2026.' additional_kwargs={'reasoning_content': 'The user asks: "What year is it?" The system says current date: 2026-09-08. So answer: 2026. Probably mention it\'s 2026.'} response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 76, 'total_tokens': 132, 'completion_time': 0.080254244, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003765007, 'prompt_tokens_details': None, 'queue_time': 0.172965062, 'total_time': 0.084019251}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a07f41-6a09-7782-8f29-8cc88e5fecff-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 76, 'output_tokens': 56, 'total_tokens': 132, 'output_token_details': {'reasoning': 39}}


In [62]:
prompt = Client().pull_prompt("hwchase17/react",dangerously_pull_public_prompt=True)

In [63]:
tools = [search_tool]

In [64]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [72]:
#AGENT 

agent = create_react_agent(
    model=llm, #brain
    tools=tools, #the tools that the agent can use to answer questions
    debug=True
)

/var/folders/cy/y01fd47x42727vv025fnkyhw0000gn/T/ipykernel_88316/3488171631.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [73]:
result = agent.invoke({
    "messages": [("user", "What is the capital of France?")]
})
print(result["messages"][-1].content)


[values] {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='a5c48e62-e21f-489a-b383-d81bb0745ef1')]}
[updates] {'agent': {'messages': [AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks: "What is the capital of France?" This is a straightforward factual question. The answer: Paris. No need for search.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 161, 'total_tokens': 209, 'completion_time': 0.050034024, 'completion_tokens_details': {'reasoning_tokens': 30}, 'prompt_time': 0.007934388, 'prompt_tokens_details': None, 'queue_time': 0.130160579, 'total_time': 0.057968412}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07f50-7508-7681-97a1-822b4aec7de6-0', tool_calls=[], invalid_tool_calls=[]

In [74]:
result = agent.invoke({
    "messages": [("user", "What is the capital of France?"
    " and what is the current weather there?")]
})
print(result["messages"][-1].content)


[values] {'messages': [HumanMessage(content='What is the capital of France? and what is the current weather there?', additional_kwargs={}, response_metadata={}, id='035dc212-ad44-4d81-b639-c426b3a924c8')]}
[updates] {'agent': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: capital of France is Paris. Current weather: need up-to-date info. Use search.', 'tool_calls': [{'id': 'fc_02265e09-c250-44dd-848a-b60488dc3141', 'function': {'arguments': '{"query":"current weather in Paris"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 169, 'total_tokens': 223, 'completion_time': 0.056881246, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.009824435, 'prompt_tokens_details': None, 'queue_time': 0.239580764, 'total_time': 0.066705681}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_deman

In [75]:
result = agent.invoke({
    "messages": [("user", "What is the latest news about USA IRAN war?")]
})
print(result["messages"][-1].content)


[values] {'messages': [HumanMessage(content='What is the latest news about USA IRAN war?', additional_kwargs={}, response_metadata={}, id='aa5aa38f-5f69-4da0-95a8-31907b387e49')]}
[updates] {'agent': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the latest news about USA IRAN war?" They want the latest news. We need to browse. Use the search tool.', 'tool_calls': [{'id': 'fc_c2cbd54e-1ef3-4207-93c4-f10ea3f611ad', 'function': {'arguments': '{"query":"latest news USA Iran war"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 165, 'total_tokens': 229, 'completion_time': 0.065703615, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.007955986, 'prompt_tokens_details': None, 'queue_time': 0.081932392, 'total_time': 0.073659601}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_d

In [77]:
#say we wantto use a custom tool, we can create a custom tool and add it to the tools list. For example, we can create a custom tool that returns the current date and time.

#for example getting the weather 
@tool
def get_current_weather(location: str) -> str:

    url =(
        f"https://api.weatherstack.com/current"
        f"?access_key={WEATHER_STACK_API_KEY}&query={location}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not retrieve weather data for {location}. Please check the location and try again."
    else:
        temperature = data["current"]["temperature"]
        weather_descriptions = data["current"]["weather_descriptions"][0]
        return f"The current temperature in {location} is {temperature}°C with {weather_descriptions}."


ValueError: Function must have a docstring if description not provided.